# KrishiMitr — Real Leaf Disease Model Training
Run the cells in order. In Colab select **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
import torch, torchvision
assert torch.cuda.is_available(), 'GPU is not enabled. Go to Runtime > Change runtime type > T4 GPU, then run again.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
# Download the official open PlantVillage dataset (about 1.8 GB).
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/PlantVillage-Dataset
DATA_SOURCE = '/content/PlantVillage-Dataset/raw/color'

In [ ]:
# Create the 80% training / 20% validation split.
from pathlib import Path
import random, shutil
source, output = Path(DATA_SOURCE), Path('/content/plant_disease')
random.seed(42)
for class_dir in sorted(path for path in source.iterdir() if path.is_dir()):
    images = [path for path in class_dir.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    random.shuffle(images)
    val_count = max(1, round(len(images) * 0.20))
    for split, items in [('val', images[:val_count]), ('train', images[val_count:])]:
        folder = output / split / class_dir.name
        folder.mkdir(parents=True, exist_ok=True)
        for image in items:
            shutil.copy2(image, folder / image.name)
print('Dataset ready:', len(list((output / 'train').iterdir())), 'classes')

In [ ]:
# Train EfficientNet-B0 using transfer learning. About 20–60 minutes on a T4 GPU.
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
device = torch.device('cuda')
train_tf = transforms.Compose([transforms.Resize((256, 256)), transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(), transforms.RandomRotation(12), transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.12), transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
val_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
train_set = datasets.ImageFolder('/content/plant_disease/train', transform=train_tf)
val_set = datasets.ImageFolder('/content/plant_disease/val', transform=val_tf)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(train_set.classes))
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
best = 0.0
for epoch in range(1, 11):
    model.train(); correct = total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(); logits = model(images); loss = loss_fn(logits, labels); loss.backward(); optimizer.step()
        correct += (logits.argmax(1) == labels).sum().item(); total += labels.size(0)
    model.eval(); val_correct = val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            logits = model(images.to(device)); val_correct += (logits.argmax(1).cpu() == labels).sum().item(); val_total += labels.size(0)
    train_acc, val_acc = correct / total, val_correct / val_total
    print(f'Epoch {epoch}/10 — train: {train_acc:.1%}, validation: {val_acc:.1%}')
    if val_acc > best:
        best = val_acc
        torch.save({'state_dict': model.state_dict(), 'class_labels': train_set.classes}, '/content/efficientnet_leaf_disease.pt')
        print('Saved best model:', f'{best:.1%}')

In [ ]:
# Download the finished model to your computer.
from google.colab import files
files.download('/content/efficientnet_leaf_disease.pt')